# EEG_01: Introduction to EEG, Neurophysiology & MESA Sleep Dataset Audit
*(Giai đoạn 1 – Tuần 1: Sinh lý Thần kinh, Cơ chế Vật lý Điện não và Khai phá Dữ liệu MESA Sleep)*

---

## 1. Mục tiêu Thực nghiệm (Learning & Research Objectives)
1. **Mô phỏng Vật lý Thần kinh (Biophysical Modeling):** Chứng minh bằng toán học và mô phỏng số tại sao điện não đồ bề mặt (Scalp EEG) phản ánh sự cộng hưởng của **Điện thế Hậu Synap (Postsynaptic Potentials - EPSP/IPSP, $15\text{–}100\text{ ms}$)** từ các tế bào tháp vỏ não (Pyramidal Neurons), chứ không phải **Điện thế Hoạt động (Action Potentials, $<2\text{ ms}$)**.
2. **Phân tích 5 Dải tần số EEG Chuẩn & Quy luật Phổ $1/f^\alpha$:** Xây dựng và phân tích mật độ phổ công suất (Welch PSD) của các nhịp **Delta ($\delta$), Theta ($\theta$), Alpha ($\alpha$), Beta ($\beta$), Gamma ($\gamma$)** ở tần số lấy mẫu chuẩn của bộ dữ liệu MESA ($f_s = 256\text{ Hz}$).
3. **Đạo trình Điện cực (Montages), Khử nhiễu Đồng pha (CMRR) & Nhiễu EEG:** Kiểm chứng sự khác biệt giữa đạo trình đơn cực (**Monopolar `EEG3`: $\text{C4}-\text{M1}$**) và lưỡng cực (**Bipolar `EEG1`: $\text{Fz}-\text{Cz}$, `EEG2`: $\text{Cz}-\text{Oz}$**) trước các nguồn nhiễu sinh lý (EOG chớp mắt, EMG cơ cằm) và nhiễu điện lưới $60\text{ Hz}$.
4. **Khai phá & Kiểm toán Dữ liệu MESA Sleep (EDF + NSRR XML Parser):** Cung cấp bộ đọc chuẩn hóa tự động kiểm tra đơn vị vật lý ($\mu\text{V}$), tần số lấy mẫu, trích xuất các kênh PSG và đồng bộ hóa nhãn 5 giai đoạn ngủ (Wake, N1, N2, N3, REM) cùng sự kiện hô hấp (Apnea/Hypopnea) và vi thức tỉnh (Cortical Arousal).

In [ ]:
import os
import glob
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import matplotlib as mpl

# Chuẩn định dạng đồ thị khoa học (IEEE / Nature / PLOS ONE)
plt.style.use('default')
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'axes.titleweight': 'bold',
    'axes.grid': False,
    'axes.linewidth': 0.8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

def clean_spines(ax):
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color('#94A3B8')
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_color('#94A3B8')
    ax.spines['bottom'].set_linewidth(0.8)

np.random.seed(42)
print('Environment initialized successfully.')

## 2. Thực nghiệm 1: Mô phỏng Cộng hưởng Không gian - Thời gian của Action Potentials vs. Postsynaptic Potentials

Điện thế ngoại bào tại vị trí điện cực $\mathbf{r}_e$ do một quần thể $N$ tế bào thần kinh tạo ra trong môi trường dẫn điện đồng nhất (độ dẫn điện $\sigma$) tuân theo phương trình Poisson dạng tích phân:
$$V_e(\mathbf{r}_e, t) = \frac{1}{4\pi\sigma} \sum_{k=1}^{N} \sum_{j} \frac{I_{k,j}(t)}{|\mathbf{r}_e - \mathbf{r}_{k,j}|}$$

Trong đó, mỗi đáp ứng hậu synap kích thích (EPSP) được mô hình hóa bằng hàm hai hằng số thời gian (Bi-exponential / Alpha synapse):
$$h_{\text{PSP}}(t) = A \left( e^{-t / \tau_d} - e^{-t / \tau_r} \right) u(t), \quad \tau_r \approx 2\text{ ms}, \; \tau_d \approx 25\text{ ms}$$

In [ ]:
fs_micro = 10000  # 10 kHz để phân giải được xung Action Potential (< 2 ms)
dur = 0.5
t_micro = np.arange(0, dur, 1/fs_micro)
n_samples = len(t_micro)
n_neurons = 200

# 1. Kernel của Action Potential (Xung hai pha cực nhanh ~1.5 ms)
t_k_ap = np.arange(-0.005, 0.005, 1/fs_micro)
kernel_ap = -np.sin(2 * np.pi * 600 * t_k_ap) * np.exp(-(t_k_ap / 0.0008)**2)

# 2. Kernel của EPSP (tau_rise = 2 ms, tau_decay = 25 ms)
t_k_psp = np.arange(0, 0.150, 1/fs_micro)
tau_r, tau_d = 0.002, 0.025
kernel_psp = np.exp(-t_k_psp / tau_d) - np.exp(-t_k_psp / tau_r)
kernel_psp /= np.max(kernel_psp)

# 3. Quá trình phóng điện Poisson không thuần nhất điều chế theo nhịp Alpha 10 Hz
rate_modulation = 0.5 * (1 + np.cos(2 * np.pi * 10 * t_micro))
sum_ap = np.zeros(n_samples)
sum_psp = np.zeros(n_samples)

for _ in range(n_neurons):
    spikes = (np.random.rand(n_samples) < (rate_modulation * 40.0 / fs_micro)).astype(float)
    sum_ap += np.convolve(spikes, kernel_ap, mode='same')
    sum_psp += np.convolve(spikes, kernel_psp, mode='full')[:n_samples]

idx_s = int(0.05 * fs_micro)
t_plot = (t_micro[idx_s:] - 0.05) * 1000
sum_ap_s = (sum_ap[idx_s:] - np.mean(sum_ap[idx_s:])) / np.std(sum_ap[idx_s:]) * 8.0
sum_psp_s = (sum_psp[idx_s:] - np.mean(sum_psp[idx_s:])) / np.std(sum_psp[idx_s:]) * 32.0

fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.2), sharex=True, facecolor='white')
axes[0].plot(t_plot, sum_ap_s, color='#64748B', lw=0.85, label='Summed Action Potentials (~1 ms biphasic spikes)')
axes[0].set_ylabel(r'Extracellular $V_e$ [a.u.]')
axes[0].set_title('(a) Asynchronous Summation of 200 Action Potentials (Phase Cancellation)', loc='left')
clean_spines(axes[0])
axes[0].legend(loc='upper right', frameon=False)

axes[1].plot(t_plot, sum_psp_s, color='#2563EB', lw=1.5, label='Summed Postsynaptic Potentials (EPSPs, 15–40 ms)')
axes[1].set_xlabel('Time [ms]')
axes[1].set_ylabel(r'Scalp Potential [$\mu$V]')
axes[1].set_title('(b) Temporal Integration of 200 EPSPs Generates Macroscopic 10 Hz Alpha Wave', loc='left')
clean_spines(axes[1])
axes[1].legend(loc='upper right', frameon=False)
plt.tight_layout()
plt.show()

## 3. Thực nghiệm 2: Pipeline Kiểm toán & Trích xuất Bộ dữ liệu MESA Sleep (EDF + NSRR XML)

Hàm dưới đây được thiết kế để:
- Tự động tìm và đọc file `.edf` và `*-nsrr.xml` trong thư mục dữ liệu MESA của bạn (ví dụ `E:/MESA/` hoặc thư mục bạn chỉ định).
- Kiểm tra nghiêm ngặt **tần số lấy mẫu ($f_s = 256\text{ Hz}$ cho EEG/EOG/EMG/ECG/Pleth)** và **đơn vị vật lý ($\mu\text{V}$)**.
- Trong trường hợp bạn đang chạy thử nghiệm trên máy chưa gắn ổ cứng chứa file `.edf` dung lượng lớn, pipeline sẽ tự động tạo một **Bản ghi MESA Giả lập Chuẩn Vật lý (Physiologically Realistic MESA Emulator)** có đầy đủ các kênh `EEG1`, `EEG2`, `EEG3`, `E1`, `E2`, `EMG`, `EKG`, `Pleth`, `SpO2` và cấu trúc cây XML chuẩn NSRR để mọi ô code phía sau đều chạy mượt mà 100%.

In [ ]:
def parse_mesa_nsrr_xml(xml_path, epoch_sec=30):
    """
    Giải mã file nhãn *-nsrr.xml chuẩn của MESA Sleep Dataset.
    Trả về:
      - hypnogram: mảng numpy chứa nhãn từng epoch 30s (0:W, 1:N1, 2:N2, 3:N3, 4:REM, -1:Unscored)
      - events_df: DataFrame chứa các sự kiện hô hấp (Apnea/Hypopnea) và vi thức tỉnh (Arousal)
    """
    stage_map = {
        'Wake|0': 0,
        'Stage 1 sleep|1': 1,
        'Stage 2 sleep|2': 2,
        'Stage 3 sleep|3': 3,
        'Stage 4 sleep|4': 3,  # Gộp S3 và S4 thành N3 theo chuẩn AASM
        'REM sleep|5': 4,
        'Unscored|9': -1,
        'Movement|6': -1
    }
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    stage_events = []
    clinical_events = []
    max_end_sec = 0.0
    
    for ev in root.findall('.//ScoredEvent'):
        ev_type = ev.findtext('EventType', default='')
        ev_concept = ev.findtext('EventConcept', default='')
        start = float(ev.findtext('Start', default='0'))
        duration = float(ev.findtext('Duration', default='0'))
        end_sec = start + duration
        if end_sec > max_end_sec:
            max_end_sec = end_sec
            
        if 'Stages' in ev_type or ev_concept in stage_map:
            stage_events.append((start, duration, stage_map.get(ev_concept, -1)))
        else:
            desat = ev.findtext('Desaturation', default=None)
            clinical_events.append({
                'EventType': ev_type,
                'EventConcept': ev_concept,
                'Start_s': start,
                'Duration_s': duration,
                'Desaturation': float(desat) if desat is not None else np.nan
            })
            
    n_epochs = int(np.ceil(max_end_sec / epoch_sec))
    hypnogram = np.full(n_epochs, -1, dtype=int)
    for start, duration, label in stage_events:
        ep_start = int(round(start / epoch_sec))
        ep_count = int(round(duration / epoch_sec))
        hypnogram[ep_start : min(ep_start + ep_count, n_epochs)] = label
        
    return hypnogram, pd.DataFrame(clinical_events)

print('MESA NSRR XML Parser defined successfully.')

## 4. Bài tập Thực hành (Practical Exercises)
1. **Bài tập 1 (Kiểm toán Đạo trình MESA):** Trỏ biến `MESA_DATA_DIR` tới thư mục chứa các file `.edf` và `.xml` của bạn, chạy hàm kiểm toán trên 5 bệnh nhân đầu tiên để lập bảng thống kê: thời lượng ghi (giờ), tỷ lệ phần trăm từng giai đoạn ngủ (W, N1, N2, N3, REM) và kiểm tra xem có kênh `EEG1`, `EEG2`, `EEG3` nào bị bão hòa biên độ ($> 500\text{ }\mu\text{V}$) không.
2. **Bài tập 2 (Tỷ số Phổ Alpha Chẩm so với Trán):** Tính mật độ phổ công suất Welch trên dải Alpha ($8\text{–}12\text{ Hz}$) ở giai đoạn **Wake** và **N1** trên hai kênh `EEG1` ($\text{Fz}-\text{Cz}$, vùng trán - trung tâm) và `EEG2` ($\text{Cz}-\text{Oz}$, vùng trung tâm - chẩm). Chứng minh bằng số liệu rằng sự suy giảm công suất Alpha khi chuyển từ Wake sang N1 thể hiện rõ nhất ở kênh `EEG2` có điện cực chẩm $\text{Oz}$.